---
title: "ERCOT Price Snapshot"
execute:
  enabled: true
  freeze: false
---

In [ ]:
import pandas as pd
from IPython.display import display

# This notebook is in projects/, one level below data/.
prices = pd.read_csv("../data/ercot_spp.csv")
print(f"Downloaded {len(prices):,} observations")
display(prices.head())

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm

RTCB_GO_LIVE = pd.Timestamp("2025-12-05")  # RTC+B went live Dec 5, 2025
TZ = "America/Chicago"


def plot_rtcb(prices, location=None):
    df = prices.copy()
    ts = pd.to_datetime(df["interval_start_local"], utc=True, errors="coerce")
    df["ts"] = ts.dt.tz_convert(TZ).dt.tz_localize(None)
    df = df.dropna(subset=["ts", "spp"])

    loc = location or df["location"].value_counts().idxmax()
    d = df[df["location"] == loc].copy()
    d["date"] = d["ts"].dt.normalize()
    d["hour"] = d["ts"].dt.hour
    d["era"] = np.where(d["date"] >= RTCB_GO_LIVE, "After RTC+B", "Before RTC+B")
    colors = {"Before RTC+B": "#7aa2ff", "After RTC+B": "#ff5c8a"}
    heat = d.pivot_table(index="hour", columns="date", values="spp", aggfunc="mean")

    with plt.style.context("dark_background"):
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(14, 9), facecolor="#0b0f19",
            gridspec_kw={"height_ratios": [1.3, 1]},
        )
        for ax in (ax1, ax2):
            ax.set_facecolor("#0b0f19")
            ax.spines[["top", "right"]].set_visible(False)

        # Top: date x hour heatmap
        x = heat.columns.values
        vmax = max(300, np.nanpercentile(heat.values, 99))
        im = ax1.pcolormesh(
            x, np.arange(24), heat.reindex(range(24)).values, cmap="magma",
            shading="nearest", norm=SymLogNorm(linthresh=30, vmin=-50, vmax=vmax),
        )
        ax1.set_ylim(23.5, -0.5)
        ax1.set_yticks(range(0, 24, 3))
        ax1.set_yticklabels([f"{h:02d}:00" for h in range(0, 24, 3)])
        ax1.set_ylabel("Hour of day (CT)")
        ax1.set_title(f"ERCOT real-time SPP at {loc}  |  hour of day x date",
                      loc="left", fontsize=15, fontweight="bold")
        if x.min() <= RTCB_GO_LIVE <= x.max():
            ax1.axvline(RTCB_GO_LIVE, color="#00f0c8", lw=2, ls="--")
            ax1.text(RTCB_GO_LIVE, 22.8, "  RTC+B go-live", color="#00f0c8",
                     fontsize=11, va="center", ha="left", fontweight="bold",
                     bbox=dict(facecolor="#0b0f19", alpha=0.75, edgecolor="none", pad=2))
        fig.colorbar(im, ax=ax1, pad=0.01, fraction=0.025).set_label("$/MWh (symlog)")

        # Bottom: typical daily shape, before vs after
        for era, g in d.groupby("era"):
            prof = g.groupby("hour")["spp"].quantile([0.1, 0.5, 0.9]).unstack()
            ax2.fill_between(prof.index, prof[0.1], prof[0.9], color=colors[era], alpha=0.18)
            ax2.plot(prof.index, prof[0.5], color=colors[era], lw=3,
                     label=f"{era} (median, p10-p90)")
        ax2.set_xlim(0, 23)
        ax2.set_xticks(range(0, 24, 2))
        ax2.set_xlabel("Hour of day (CT)")
        ax2.set_ylabel("$/MWh")
        ax2.set_title("Typical daily price shape, before vs after", loc="left", fontsize=13)
        ax2.grid(alpha=0.15)
        ax2.legend(frameon=False)

        fig.tight_layout()
        plt.show()
    return fig


fig = plot_rtcb(prices)                 # busiest location
# fig = plot_rtcb(prices, "HB_HOUSTON")  # or pick one
# fig.savefig("rtcb.png", dpi=200, facecolor=fig.get_facecolor())

NameError: name 'prices' is not defined